### Data

In [ ]:
capacity=[130, 185, 152, 290, 137, 84, 183, 275]
fixedcost=[192, 290, 285, 411, 230, 95, 223, 482]
cost = [[1.53, 0.9199999999999999, 1.73, 0.9199999999999999, 1.38, 0.88, 0.86, 1.93, 0.85, 0.86], [1.61, 0.76, 1.82,
0.9099999999999999, 1.46, 0.74, 0.81, 2.0300000000000002, 0.76, 0.75], [1.55, 0.87, 1.76, 0.9099999999999999, 1.4,
0.8400000000000001, 0.8400000000000001, 1.96, 0.8200000000000001, 0.8200000000000001], [1.44, 1.3, 1.6, 1.13, 1.33,
1.25, 1.1600000000000001, 1.78, 1.19, 1.21], [1.49, 1.03, 1.68, 0.96, 1.35, 0.99, 0.94, 1.88, 0.95, 0.96], [1.45, 1.51,
1.59, 1.3, 1.37, 1.46, 1.35, 1.74, 1.3900000000000001, 1.4100000000000001], [1.53, 0.9199999999999999, 1.73,
0.9199999999999999, 1.38, 0.89, 0.87, 1.93, 0.86, 0.86], [1.48, 1.63, 1.59, 1.4100000000000001, 1.4100000000000001,
1.58, 1.46, 1.73, 1.5, 1.52]]
unmet_pen = 5

demandmean = [8, 53, 79, 123, 27, 83, 17, 33, 41, 47]
demandstdev = [2,20,28,38,5,20,6,9,14,22]

10-element Vector{Int64}:
  2
 20
 28
 38
  5
 20
  6
  9
 14
 22

In [395]:
using JuMP, Gurobi, Random, Distributions, LinearAlgebra
function read_orlib_cap(filepath::String)
    open(filepath) do io
        # 1) read n (#facilities) and m (#cities)
        n, m = parse.(Int, split(readline(io)))

        # 2) read facility data: capacity and opening cost
        capacity  = Vector{Float64}(undef, n)
        fixedcost = Vector{Float64}(undef, n)
        for i in 1:n
            # each line: "<capacity> <opening cost>"
            c1, c2 = parse.(Float64, split(readline(io)))
            capacity[i], fixedcost[i] = c1, c2
        end

        # 3) read city‐by‐city: demand + cost to each facility
        demand = Vector{Float64}(undef, m)
        cost = Array{Float64}(undef, n, m)
        for j in 1:m
            # read demand line
            demand[j] = parse(Float64, strip(readline(io)))
            # now read cost entries until we have n of them
            vals = Float64[]
            while length(vals) < n
                append!(vals, parse.(Float64, split(readline(io))))
            end
            cost[:,j] = vals[1:n]
        end

        return capacity, fixedcost, cost, demand
    end
end

# ── Reading files ──
file_path = "/Users/Patron/Documents/ORLIB/ORLIB-cap/90/cap91.txt"
capacity, fixedcost, cost, demandmean = read_orlib_cap(file_path);


In [396]:
Random.seed!(1234)      # for reproducibility
nloc = length(fixedcost)

# choose your perturbation magnitude (say ±10%)
delta = 0.5
dist_eps = Uniform(-delta, delta)
caps_eps = Uniform(-delta,delta)

# draw one epsilon per facility
eps = rand(dist_eps, nloc)        # vector of length nloc
eps1 = rand(caps_eps, nloc)        # vector of length nloc

# form the perturbed fixed‐costs
fixedcost_perturbed = fixedcost .* (1 .+ eps)
capacity_perturbed = fixedcost.*(1 .+ eps1)

# if you want integers:
fixedcost = round.(Int, fixedcost_perturbed)
capacity = round.(Int, capacity_perturbed)


25-element Vector{Int64}:
  7634
  9373
  7894
  4376
  7693
 10055
  7680
  3846
  7371
 10626
     ⋮
  5515
  9871
  4383
  8485
 10012
 10348
  9395
  4724
  6556

In [397]:
theta = rand(Uniform(0.5, 1), 50)
#theta = 1
demandstdev = theta .* demandmean;

In [184]:
mean(demandmean)


1165.36

In [219]:
mean(demandstdev)

3357.579242154219

### Facility Location 

In [398]:
function facility_location(nloc,ncust,capacity,cost, demand)
    model = Model(Gurobi.Optimizer)
    # Decision Variables
    @variable(model,x[1:nloc], Bin)
    @variable(model,y[1:nloc,1:ncust]>=0)
    @variable(model,shortfall[1:ncust]>=0)

    #constraints
    @constraint(model,[j in 1:ncust], sum(y[i,j] for i in 1:nloc) + shortfall[j] == demand[j])
    @constraint(model,[i in 1:nloc], sum(y[i,j] for j in 1:ncust) - capacity[i]*x[i]<=0 )

    unmet_pen = 15000

    #Objective function
    @objective(model, Min, sum(fixedcost[i]*x[i] for i in 1:nloc) + sum(cost[i,j]y[i,j] for i in 1:nloc for j in 1:ncust) + sum(unmet_pen*shortfall[j] for j in 1:ncust))

    optimize!(model)

    return objective_value(model),collect(value.(x))
end

a,b= facility_location(25,50,capacity,cost,demandmean)




(6.6439056005e8, [1.0, -0.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0  …  1.0, -0.0, -0.0, 1.0, 1.0, 1.0, 1.0, -0.0, 1.0, 1.0])

### Sampling

In [399]:
function sampling(nscen,nloc,demandmean,demandstdev)
    ncust = length(demandmean)
    xsolutions = zeros(nscen,nloc)
    demand = zeros(ncust)
    for scenario in 1:nscen
        globaldem = round(max(rand(Normal(1000,900)),0),digits=0)
        for i in 1:ncust
            demand[i]=globaldem+round(max(rand(Normal(demandmean[i],demandstdev[i])),0),digits=0)
        end
        _,xsolutions[scenario,:] = facility_location(nloc,ncust,capacity,cost,demand)
    end
    return xsolutions
end
xsolutions = sampling(50,25,demandmean,demandstdev);

In [367]:
xsolutions

50×25 Matrix{Float64}:
 1.0  -0.0   1.0  1.0  1.0  1.0  1.0  1.0  …   1.0  1.0  1.0  -0.0   1.0  1.0
 1.0  -0.0   1.0  1.0  1.0  1.0  1.0  1.0      1.0  1.0  1.0  -0.0   1.0  1.0
 1.0  -0.0   1.0  1.0  1.0  1.0  1.0  1.0      1.0  1.0  1.0  -0.0   1.0  1.0
 1.0   0.0   1.0  1.0  1.0  1.0  1.0  1.0      1.0  1.0  1.0   0.0   1.0  1.0
 1.0   0.0   1.0  1.0  1.0  1.0  1.0  1.0      1.0  1.0  1.0   0.0   1.0  1.0
 1.0  -0.0   1.0  1.0  1.0  1.0  1.0  1.0  …   1.0  1.0  1.0  -0.0   1.0  1.0
 1.0  -0.0   1.0  1.0  1.0  1.0  1.0  1.0      1.0  1.0  1.0  -0.0   1.0  1.0
 1.0  -0.0   1.0  1.0  1.0  1.0  1.0  1.0      1.0  1.0  1.0  -0.0   1.0  1.0
 1.0  -0.0   1.0  1.0  1.0  1.0  1.0  1.0      1.0  1.0  1.0  -0.0   1.0  1.0
 1.0  -0.0   1.0  1.0  1.0  1.0  1.0  1.0      1.0  1.0  1.0  -0.0   1.0  1.0
 ⋮                          ⋮              ⋱        ⋮                     
 1.0  -0.0   1.0  1.0  1.0  1.0  1.0  1.0      1.0  1.0  1.0  -0.0   1.0  1.0
 1.0  -0.0   1.0  1.0  1.0  1.0  1.0  1.0   

In [402]:
function n_unique_rows(A::AbstractMatrix)
    # turn every row into an immutable tuple
    rows_as_tuples = (Tuple(view(A, i, :)) for i in axes(A,1))
    uniq = unique(rows_as_tuples)     # removes duplicates
    return uniq, length(uniq)
end
u,_= n_unique_rows(xsolutions)
u

10-element Vector{NTuple{25, Float64}}:
 (1.0, -0.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 0.0, 1.0, -0.0, 1.0, 1.0, 1.0, -0.0, -0.0, 1.0, 1.0, 1.0, 1.0, -0.0, 1.0, 1.0)
 (1.0, -0.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 0.0, 1.0, -0.0, 1.0, 1.0, 1.0, -0.0, 1.0, 1.0, 1.0, 1.0, 1.0, -0.0, 1.0, 1.0)
 (1.0, -0.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 0.0, 1.0, -0.0, 1.0, 1.0, 1.0, -0.0, 0.0, 1.0, 1.0, 1.0, 1.0, -0.0, 1.0, 1.0)
 (1.0, -0.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, -0.0, 1.0, 0.0, 1.0, -0.0, 1.0, 1.0, -0.0, -0.0, -0.0, 1.0, 1.0, 1.0, 1.0, -0.0, 1.0, 1.0)
 (1.0, -0.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 0.0, 1.0, -0.0, 1.0, 1.0, 1.0, -0.0, -0.0, -0.0, 1.0, 1.0, 1.0, -0.0, 1.0, 1.0)
 (1.0, 0.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 0.0, 1.0, 1.0, 1.0, 1.0, 1.0, 0.0, 1.0, 1.0, 1.0, 1.0, 1.0, 0.0, 1.0, 1.0)
 (1.0, -0.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 0.0, 1.0, -0.0, 1.0, 1.0, 1.0, -0.0, -0.0, 1.0, 1.0, 1.0, 1.0, 0.0, 1.0, 1.0)
 (1.0, -0.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0

In [382]:
nsamples = 100
ncust = 50
online_samples = zeros(nsamples, ncust)
for s in 1:nsamples
    globaldem = round(max(rand(Normal(1000,900)),0),digits=0)
    for j in 1:ncust
        online_samples[s,j] = globaldem+round(max(rand(Normal(demandmean[j],demandstdev[j])),0),digits=0)
    end
end

In [383]:
online_samples

100×50 Matrix{Float64}:
 1117.0  1179.0  1690.0  1003.0  1020.0  …  1003.0  1213.0  1639.0  1518.0
 1802.0  1978.0  1753.0  3157.0  1785.0     2266.0  1753.0  2011.0  2061.0
 2677.0  2714.0  2677.0  4784.0  2682.0     3202.0  2677.0  2677.0  2677.0
 2539.0  2549.0  2516.0  3892.0  2519.0     2516.0  2599.0  2516.0  3216.0
  574.0   550.0  2553.0  2405.0   477.0      979.0   535.0   477.0   477.0
 3376.0  3228.0  3509.0  4642.0  3150.0  …  3306.0  3148.0  3148.0  3814.0
  665.0   449.0   677.0   449.0   469.0      449.0   493.0  2909.0   449.0
 1293.0  1553.0  1909.0  3462.0  1293.0     1318.0  1301.0  4026.0  1945.0
  373.0     8.0  1497.0  3555.0    11.0      387.0     0.0  1834.0   389.0
  662.0   498.0   467.0   467.0   482.0      467.0   517.0  5376.0  1206.0
    ⋮                                    ⋱                          
 1346.0  1281.0  3337.0  1197.0  1227.0     1263.0  1197.0  1197.0  1428.0
 1012.0   733.0  1016.0  3688.0   786.0      733.0   936.0  4791.0   881.0
  759.0

### Decoupling of continuous and binary variable

In [384]:
function solve_for_continuous_var(nloc,ncust,capacity,cost,demand,solved_x)
    model = Model(Gurobi.Optimizer)
    # Decision Variables
    @variable(model,x[1:nloc], Bin)
    @variable(model,y[1:nloc,1:ncust]>=0)
    @variable(model,shortfall[1:ncust]>=0)

    #constraints
    @constraint(model,[j in 1:ncust], sum(y[i,j] for i in 1:nloc) + shortfall[j] == demand[j])
    @constraint(model,[i in 1:nloc], sum(y[i,j] for j in 1:ncust) - capacity[i]*x[i]<=0 )
    @constraint(model, x == solved_x)  # fixing the binary decision variable

    unmet_pen = 15000

    #Objective function
    @objective(model, Min, sum(fixedcost[i]*x[i] for i in 1:nloc) + sum(cost[i,j]y[i,j] for i in 1:nloc for j in 1:ncust) + sum(unmet_pen*shortfall[j] for j in 1:ncust))

    optimize!(model)

    return objective_value(model), value.(y)
end
#y = solve_for_continuous_var(8,10,capacity,cost,demandmean, xsolutions[1,:]) 
function Calculations_For_Extensive_Form(nloc,ncust,capacity,cost,online_samples, xsolutions)
    total_scenarios = size(online_samples,1)
    total_solutions = size(xsolutions,1)
     obj_value = zeros(total_scenarios,total_solutions)
    for scenario in 1:total_scenarios
        demand = online_samples[scenario,:]
        for solution in 1:total_solutions
            obj_value[scenario,solution], _ = solve_for_continuous_var(nloc,ncust,capacity,cost,demand,xsolutions[solution,:])
        end
    end
    return obj_value
end

obj_matrix = Calculations_For_Extensive_Form(25,50,capacity,cost,online_samples,xsolutions)


100×50 Matrix{Float64}:
 1.09505e9  1.09506e9  1.09505e9  …  1.09505e9  1.09506e9  1.09507e9
 1.15678e9  1.15679e9  1.15678e9     1.15678e9  1.15679e9  1.15678e9
 1.46353e9  1.45906e9  1.46353e9     1.46353e9  1.45906e9  1.45724e9
 1.48451e9  1.48137e9  1.48451e9     1.48451e9  1.48137e9  1.48057e9
 1.06071e9  1.06072e9  1.06071e9     1.06071e9  1.06072e9  1.06073e9
 1.81953e9  1.80056e9  1.81953e9  …  1.81953e9  1.80056e9  1.79337e9
 7.20205e8  7.20211e8  7.20205e8     7.20205e8  7.20211e8  7.20219e8
 1.52311e9  1.52295e9  1.52311e9     1.52311e9  1.52295e9  1.52296e9
 1.16169e9  1.1617e9   1.16169e9     1.16169e9  1.1617e9   1.1617e9
 7.54476e8  7.54482e8  7.54476e8     7.54476e8  7.54482e8  7.5449e8
 ⋮                                ⋱                        
 1.22717e9  1.22718e9  1.22717e9     1.22717e9  1.22718e9  1.22719e9
 7.09565e8  7.09571e8  7.09565e8     7.09565e8  7.09571e8  7.0958e8
 1.02706e9  1.02706e9  1.02706e9     1.02706e9  1.02706e9  1.02707e9
 1.29618e9  1.29619e9 

### Reformulated Extensive Form

In [385]:
function Reformulated_Extensive_Form(S,l,K,obj_value)
    # Initialize model
    model = Model(Gurobi.Optimizer)
    MOI.set(model, MOI.TimeLimitSec(), 30.0)

    #Total scenarios

    # Decision Variables
    @variable(model, z[1:l], Bin)
    @variable(model, w[1:l , 1:S], Bin)

    # Constraints
    for scenario in 1:S
        for solution in 1:l
            if obj_value[scenario,solution] == -1
                @constraint(model, w[solution,scenario] == 0)   ## weights of infeasible scenario/first stage decision pair is 0
            end
        end
    end
            
        
    for j in 1:S
        @constraint(model, [i in 1:l], w[i,j] <= z[i])
        @constraint(model, sum(w[i,j] for i in 1:l) == 1)
    end
    @constraint(model, sum(z[i] for i in 1:l) == K)

    # Objective
    objective = 0.0
    for scenario in 1:S
        for solution in 1:l
            objective += w[solution,scenario]*obj_value[scenario,solution]
        end
    end

    # Define the objective function
    @objective(model, Min, objective/S)

    # Solve the model
    #set_silent(model)
    optimize!(model)
    # Check solver status

    if termination_status(model) == MOI.OPTIMAL
        return objective_value(model),value.(z)
    else
        return -1,[]
    end    
end 
Solution,z=Reformulated_Extensive_Form(100,50,1,obj_matrix)

(1.2757668969397492e9, [-0.0, -0.0, -0.0, -0.0, -0.0, -0.0, -0.0, -0.0, -0.0, -0.0  …  -0.0, -0.0, 1.0, 0.0, -0.0, -0.0, -0.0, -0.0, -0.0, -0.0])

### Test Sample

In [386]:
nsamples = 100
ncust = 50
test_samples = zeros(nsamples, ncust)
for s in 1:nsamples
    globaldem = round(max(rand(Normal(1000,900)),0),digits=0)
    for j in 1:ncust
        test_samples[s,j] = globaldem+round(max(rand(Normal(demandmean[j],demandstdev[j])),0),digits=0)
    end
end

### Offline Calculation of Test Data

In [387]:
function OfflineTest(nloc,ncust,capacity,cost,test_samples)
    num_rows = size(test_samples, 1)
    offline_cost= zeros(num_rows)

    #Offline calculations for the test data
    total_cost = 0.0
    for i in 1:num_rows
        obj_value,_ =facility_location(nloc,ncust,capacity,cost, test_samples[i,:])
        offline_cost[i] = obj_value
        total_cost += obj_value
    end
    total_offline_cost = total_cost/num_rows
    return total_offline_cost
end
offline_cost = OfflineTest(25,50,capacity,cost,test_samples)

1.1365220523975003e9

### Second Stage

In [389]:
function Second_Stage_Cost(nloc, ncust,cost,capacity, demand,xsolutions,z)
    model = Model(Gurobi.Optimizer)
    # Decision Variables
    @variable(model,x[1:nloc], Bin)
    @variable(model,y[1:nloc,1:ncust]>=0)
    @variable(model,shortfall[1:ncust]>=0)
    @variable(model, w[1:length(z)], Bin)

    #constraints
    @constraint(model,[j in 1:ncust], sum(y[i,j] for i in 1:nloc) + shortfall[j] == demand[j])
    @constraint(model,[i in 1:nloc], sum(y[i,j] for j in 1:ncust) - capacity[i]*x[i]<=0 )

    @constraint(model, [i in 1:length(z)], w[i] <= z[i])
    @constraint(model, sum(w[i] for i in 1:length(z)) == 1)
    @constraint(model, x == sum(w[i] * xsolutions[i,:] for i in 1:length(z)))

    unmet_pen = 15000

    #Objective function
    @objective(model, Min, sum(fixedcost[i]*x[i] for i in 1:nloc) + sum(cost[i,j]y[i,j] for i in 1:nloc for j in 1:ncust) + sum(unmet_pen*shortfall[j] for j in 1:ncust))

    optimize!(model)

    return objective_value(model)
end

    

Second_Stage_Cost (generic function with 2 methods)

### Evaluation

In [390]:
function Evaluation(nloc,ncust,cost,capacity,test_sample,x_solutions,z)
    num_rows = size(test_sample, 1)
    ### Second stage cost
    model_cost = 0.0
    for index in 1:num_rows
        model_obj_value = Second_Stage_Cost(nloc, ncust, cost,capacity,test_sample[index,:], x_solutions,z)
        model_cost += model_obj_value
    end
    total_model_cost = model_cost / num_rows

    return total_model_cost
end 
modelcost = Evaluation(25,50,cost,capacity,test_samples,xsolutions,z)

1.1368217371817503e9

In [391]:
offline_cost/100000

11365.220523975002

In [392]:
modelcost/100000

11368.217371817504

In [393]:
(offline_cost - modelcost)

-299684.784250021